In [1]:
import pandas as pd
import json
from pathlib import Path


# ============================================================
# НАСТРОЙКИ
# ============================================================

# Путь к исходному XLSX
INPUT_XLSX = Path(r"C:\Users\Ева\Desktop\Хакатон\data_preproc\dataset_info\to_convert.xlsx")

# Листы Excel.
# Если нужен первый лист — оставь 0.
SHEET_NAME = 0

# Куда сохранять JSON
OUTPUT_DIR = INPUT_XLSX.parent

ALL_JSON = OUTPUT_DIR / "all_statuses.json"
FILTERED_JSON = OUTPUT_DIR / "in_datasets_and_extra.json"


# ============================================================
# СТАТУСЫ
# ============================================================

STATUS_ALL = [
    "Есть в требованиях и в датасетах",
    "Есть в требованиях, нет в датасетах",
    "Есть в датасетах, нет в требованиях",
]

STATUS_FOR_SECOND_JSON = [
    "Есть в требованиях и в датасетах",
    "Есть в датасетах, нет в требованиях",
]


# ============================================================
# ЧТЕНИЕ XLSX
# ============================================================

df = pd.read_excel(
    INPUT_XLSX,
    sheet_name=SHEET_NAME
)

print(f"Прочитано строк: {len(df)}")
print(f"Столбцы: {list(df.columns)}")


# ============================================================
# ОЧИСТКА
# ============================================================

# Убираем полностью пустые строки
df = df.dropna(how="all").copy()

# Убираем пробелы по краям во всех строковых ячейках
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(
            lambda x: x.strip() if isinstance(x, str) else x
        )


# ============================================================
# ПРОВЕРКА СТАТУСОВ
# ============================================================

actual_statuses = set(df["Статус"].dropna().unique())

print("\nСтатусы в XLSX:")
for status in sorted(actual_statuses):
    count = (df["Статус"] == status).sum()
    print(f"  {status}: {count}")


unknown_statuses = actual_statuses - set(STATUS_ALL)

if unknown_statuses:
    print("\nВНИМАНИЕ! Обнаружены неизвестные статусы:")
    for status in unknown_statuses:
        print(f"  {status}")


# ============================================================
# ПРЕОБРАЗОВАНИЕ В JSON
# ============================================================

# Все строки
all_records = df.to_dict(orient="records")


# Только два нужных статуса
filtered_df = df[
    df["Статус"].isin(STATUS_FOR_SECOND_JSON)
].copy()

filtered_records = filtered_df.to_dict(orient="records")


# ============================================================
# СОХРАНЕНИЕ
# ============================================================

with open(ALL_JSON, "w", encoding="utf-8") as f:
    json.dump(
        all_records,
        f,
        ensure_ascii=False,
        indent=2
    )

with open(FILTERED_JSON, "w", encoding="utf-8") as f:
    json.dump(
        filtered_records,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# ИТОГ
# ============================================================

print("\nГотово!")

print(f"\n1. Все статусы:")
print(f"   {ALL_JSON}")
print(f"   Строк: {len(all_records)}")

print(f"\n2. Только нужные статусы:")
print(f"   {FILTERED_JSON}")
print(f"   Строк: {len(filtered_records)}")

print("\nРаспределение во втором JSON:")

for status in STATUS_FOR_SECOND_JSON:
    count = (filtered_df["Статус"] == status).sum()
    print(f"  {status}: {count}")

Прочитано строк: 153
Столбцы: ['Статус', 'Техника', 'Категория требования', 'Датасеты', 'Количество объектов', 'Этапы']

Статусы в XLSX:
  Есть в датасетах, нет в требованиях: 13
  Есть в требованиях и в датасетах: 13
  Есть в требованиях, нет в датасетах: 127

Готово!

1. Все статусы:
   C:\Users\Ева\Desktop\Хакатон\data_preproc\dataset_info\all_statuses.json
   Строк: 153

2. Только нужные статусы:
   C:\Users\Ева\Desktop\Хакатон\data_preproc\dataset_info\in_datasets_and_extra.json
   Строк: 26

Распределение во втором JSON:
  Есть в требованиях и в датасетах: 13
  Есть в датасетах, нет в требованиях: 13
